In [26]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

In [27]:
fj = pd.read_csv(r"C:\Users\arjun\Downloads\csv_files\fake_job_postings.csv.xls")

In [28]:
fj.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 35 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   job_id               17880 non-null  int64  
 1   title                17880 non-null  object 
 2   location             17534 non-null  object 
 3   department           6333 non-null   object 
 4   salary_range         2868 non-null   object 
 5   company_profile      14572 non-null  object 
 6   description          17879 non-null  object 
 7   requirements         15184 non-null  object 
 8   benefits             10668 non-null  object 
 9   telecommuting        17880 non-null  int64  
 10  has_company_logo     17880 non-null  int64  
 11  has_questions        17880 non-null  int64  
 12  employment_type      14409 non-null  object 
 13  required_experience  10830 non-null  object 
 14  required_education   9775 non-null   object 
 15  industry             12977 non-null 

In [29]:
fj.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,...,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
fj.shape

(17880, 35)

In [31]:
fj['x'] = (
    fj['title'].apply(str)+"  "+
    fj['location'].apply(str)+"   "+
    fj['department'].apply(str)+"  "+
    fj['company_profile'].apply(str) + "  " +
    fj['salary_range'].apply(str) + "  " +
    fj['description'].apply(str) + "  " +
    fj['requirements'].apply(str) + "  " +
    fj['benefits'].apply(str) + "  " +
    fj['industry'].apply(str) + "  "+
    fj['required_experience'].apply(str)
)

In [32]:
fj = fj.rename(columns={'fraudulent':'y'})

In [33]:
fj = fj.loc[:,['x','y']]

In [34]:
fj.head(2)

,x,y
0,"Marketing Intern US, NY, New York Marketing...",0
1,"Customer Service - Cloud Video Production NZ,...",0


In [35]:
from sklearn.model_selection import train_test_split
train,test = train_test_split(fj,test_size = .2)
fj1 = fj[fj.y == 1]
train = pd.concat([train,fj1,fj1,fj1,fj1,fj1,fj1,fj1])
fj.y.value_counts()

y
0    17014
1      866
Name: count, dtype: int64

In [36]:

train_x = train.iloc[:,0]
train_y = train.iloc[:,1]

test_x = test.iloc[:,0]
test_y = test.iloc[:,1]

In [37]:
from keras.layers import Dense , SimpleRNN,LSTM,Embedding
from keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from keras.layers import Bidirectional  # Bidirectional is used for high accuracy

In [38]:
from tensorflow.keras.utils import to_categorical
train_y = to_categorical(train_y)

In [39]:
max_num_words = 10000 # We consider up to 10,000 of the most frequent words in the entire corpus (dataset).
seq_len = 50    # how many tokens (words  to conider in each document / message)
embedding_size = 100  # vector length of each word
from tensorflow.keras.preprocessing.text import Tokenizer # assign a number to each word
from tensorflow.keras.preprocessing.sequence import pad_sequences # padding extras layers of zeros

In [40]:
tokenizer = Tokenizer(num_words = max_num_words)
tokenizer.fit_on_texts(fj.x)
train_x = tokenizer.texts_to_sequences(train_x)

In [41]:
train_x = pad_sequences(train_x, maxlen = seq_len)
# This function makes all sequences the same length by adding zeros or removing extra tokens.

In [42]:
test_x = tokenizer.texts_to_sequences(test_x)
test_x = pad_sequences(test_x, maxlen = seq_len)
# converts test text into integer sequences using the training vocabulary.
#vpad_sequences() makes all test sequences the same length so they match the input format expected by the trained NLP model.
 #   This allows us to evaluate the model consistently on unseen data.

In [43]:
model = Sequential()
model.add(Embedding(input_dim = max_num_words,
                    input_length = seq_len,
                    output_dim = embedding_size))  # Dense --> fully connected layer.
#model.add(LSTM(32))
model.add(Bidirectional(LSTM(32)))
model.add(Dense(2, activation = 'softmax')) # O/P this is u have 6 level in target variable
from tensorflow.keras.optimizers import Adam
adam = Adam(learning_rate = .001) # Adam is an optimization algorithm that updates the model's weights during training to reduce prediction errors.
model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
model.fit(train_x,train_y, epochs = 5, batch_size = 32, validation_split = .2)

Epoch 1/5
510/510 ━━━━━━━━━━━━━━━━━━━━ 31s 48ms/step - accuracy: 0.9384 - loss: 0.1705 - val_accuracy: 0.9764 - val_loss: 0.1105
Epoch 2/5
510/510 ━━━━━━━━━━━━━━━━━━━━ 41s 48ms/step - accuracy: 0.9935 - loss: 0.0220 - val_accuracy: 0.9956 - val_loss: 0.0311
Epoch 3/5
510/510 ━━━━━━━━━━━━━━━━━━━━ 24s 47ms/step - accuracy: 0.9980 - loss: 0.0076 - val_accuracy: 0.9988 - val_loss: 0.0066
Epoch 4/5
510/510 ━━━━━━━━━━━━━━━━━━━━ 42s 48ms/step - accuracy: 0.9991 - loss: 0.0027 - val_accuracy: 0.9980 - val_loss: 0.0044
Epoch 5/5
510/510 ━━━━━━━━━━━━━━━━━━━━ 25s 49ms/step - accuracy: 0.9991 - loss: 0.0028 - val_accuracy: 1.0000 - val_loss: 0.0019


In [45]:
pred = model.predict(test_x)
from sklearn.metrics import confusion_matrix
pred_cate = pred.argmax(axis = 1)
tab = confusion_matrix(test_y,pred_cate)
tab.diagonal().sum()/tab.sum()*100

112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step


np.float64(99.30089485458613)

In [46]:
from sklearn.metrics import *
print(confusion_matrix(test_y,pred_cate))
print(classification_report(test_y,pred_cate))

[[3378   25]
 [   0  173]]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      3403
           1       0.87      1.00      0.93       173

    accuracy                           0.99      3576
   macro avg       0.94      1.00      0.96      3576
weighted avg       0.99      0.99      0.99      3576



In [67]:
21/(21+3381)*100 # Fpr

0.6172839506172839

In [68]:
170/(170+4)*100 # Tpr

97.70114942528735

In [79]:
import pickle
model.save(r"C:\Users\arjun\Downloads\csv_files\fake_job_posting_RNN.keras")
with open("tokenizer.pkl","wb")as f:
  pickle.dump(tokenizer, f)

In [20]:
import os

print("Current folder:", os.getcwd())
print("Tokenizer file exists:", os.path.exists("tokenizer.pkl"))

Current folder: C:\Users\arjun
Tokenizer file exists: True


In [21]:
import os
import pickle

folder = r"C:\Users\arjun\Downloads\csv_files"

model.save(os.path.join(folder, "fake_job_posting_RNN.keras"))

with open(os.path.join(folder, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

print("Model and tokenizer saved successfully!")

Model and tokenizer saved successfully!
